# 01. Stack topology와 layer diff

## 학습 목표

- 각 branch가 바로 아래 layer를 base로 삼는지 검사합니다.
- bottom, top, upstack, downstack의 의미를 확인합니다.
- reviewer가 보는 layer별 diff를 계산합니다.

실제 GitHub나 Git branch를 변경하지 않는 Python toy simulator입니다. 위에서 아래로 실행하면 됩니다.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Layer:
    branch: str
    base: str
    own_changes: frozenset[str]
    repository: str = "example/app"

layers = [
    Layer("feature-auth", "main", frozenset({"db/user.sql", "models/user.py"})),
    Layer("feature-api", "feature-auth", frozenset({"api/auth.py", "tests/test_auth_api.py"})),
    Layer("feature-ui", "feature-api", frozenset({"ui/login.tsx", "ui/login.test.tsx"})),
]

In [ ]:
def validate_stack(layers: list[Layer], trunk: str = "main") -> None:
    if not layers:
        raise ValueError("stack에는 하나 이상의 layer가 필요합니다")
    repositories = {layer.repository for layer in layers}
    if len(repositories) != 1:
        raise ValueError("cross-repository 또는 cross-fork stack은 지원하지 않습니다")
    if layers[0].base != trunk:
        raise ValueError(f"bottom layer의 base는 {trunk}여야 합니다")
    seen = {trunk}
    for index, layer in enumerate(layers):
        expected_base = trunk if index == 0 else layers[index - 1].branch
        if layer.base != expected_base:
            raise ValueError(f"{layer.branch}: base={layer.base}, expected={expected_base}")
        if layer.branch in seen:
            raise ValueError(f"cycle 또는 중복 branch: {layer.branch}")
        seen.add(layer.branch)

validate_stack(layers)
print("bottom:", layers[0].branch)
print("top   :", layers[-1].branch)

In [ ]:
def inherited_changes(layers: list[Layer], position: int) -> frozenset[str]:
    inherited: set[str] = set()
    for layer in layers[:position]:
        inherited.update(layer.own_changes)
    return frozenset(inherited)

for position, layer in enumerate(layers):
    inherited = inherited_changes(layers, position)
    full_branch = inherited | layer.own_changes
    # PR diff는 full branch가 아니라 바로 아래 base와의 차이인 own_changes입니다.
    print(f"PR {position + 1}: {layer.branch} -> base {layer.base}")
    print("  inherited:", sorted(inherited))
    print("  review diff:", sorted(layer.own_changes))
    assert full_branch - inherited == layer.own_changes

## 잘못된 base 탐지

두 번째 layer가 main을 직접 base로 삼으면 독립 PR 두 개일 뿐 ordered stack이 아닙니다. validator가 이를 막는지 확인합니다.

In [ ]:
broken = [
    layers[0],
    Layer("feature-api", "main", frozenset({"api/auth.py"})),
]

try:
    validate_stack(broken)
except ValueError as error:
    print("예상한 오류:", error)
else:
    raise AssertionError("잘못된 base를 탐지하지 못했습니다")

print("\n확장 과제: 각 layer의 변경 파일이 겹칠 때 review 충돌 가능성을 경고하세요.")